# Bluesky and the 2024 Election: A Causal Inference Homework

**Persuasion at Scale (PSAM 3707 / UN 3707)**
Spring 2026

---

In November 2024, something remarkable happened on social media. After the U.S. presidential election, millions of people left Twitter/X and flooded into Bluesky, a newer, smaller platform. In just two weeks, Bluesky's user base nearly doubled.

This creates a natural experiment. We have a dataset of **19,243 posts** from Bluesky, collected across two major events:

1. **The 2024 U.S. Election** (Nov 5, 2024): Posts from Oct 22 - Nov 19
2. **Bluesky's Public Launch** (Feb 6, 2024): Posts from Jan 23 - Feb 20

For each event, we have posts from **before** and **after**, across four topic areas: politics, tech, sports, and culture. Each post includes engagement metrics (likes, reposts, replies) and author metadata (follower count, account age, etc.).

Your job: use causal inference to figure out what these events *did* to the platform.

### Sections

| # | Section | What you'll do |
|---|---------|---------------|
| 1 | Setup & Data Loading | Load the dataset, orient yourself |
| 2 | Getting to Know the Data | Summary statistics, distributions |
| 3 | Visualizing Engagement | Plots that tell a story |
| 4 | **Difference-in-Differences** | Estimate the causal effect of the election on political engagement |
| 5 | **Regression Discontinuity** | Look for engagement jumps at follower-count thresholds |
| 6 | Your Turn | Open-ended questions for you to investigate |

---
## Section 1: Setup & Data Loading

Run this cell to install what we need and load the dataset. Nothing to modify here.

In [ ]:
# Install packages (only needed in Colab)
!pip install -q pandas matplotlib seaborn statsmodels

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

# Make plots look nice
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

In [ ]:
# Load the dataset
DATA_URL = "https://raw.githubusercontent.com/Persuasion-at-Scale/bluesky-election-engagement/main/bluesky_posts.csv"

df = pd.read_csv(DATA_URL)

# Parse dates
df['created_at'] = pd.to_datetime(df['created_at'])
df['author_created_at'] = pd.to_datetime(df['author_created_at'], errors='coerce')

print(f"Loaded {len(df):,} posts by {df['author_did'].nunique():,} unique authors")
print(f"Columns: {list(df.columns)}")
df.head(3)

### What's in the data?

Each row is one Bluesky post. Here's what the columns mean:

| Column | What it is |
|--------|-----------|
| `post_uri` | Unique identifier for the post |
| `author_did` | Anonymous author ID |
| `author_handle` | Author's Bluesky username |
| `text` | The post content (truncated to 280 characters) |
| `created_at` | When the post was written |
| `like_count` | How many people liked this post |
| `repost_count` | How many people reposted it |
| `reply_count` | How many replies it got |
| `has_link` | 1 if the post contains a URL, 0 otherwise |
| `has_image` | 1 if the post contains an image, 0 otherwise |
| `is_reply` | 1 if this post is a reply to someone else, 0 otherwise |
| `is_quote` | 1 if this post quotes another post, 0 otherwise |
| `topic_group` | Which topic area: politics, tech, sports, or culture |
| `search_term` | The specific keyword that found this post |
| `event` | Which event window: `election_2024` or `public_launch` |
| `period` | `before` or `after` the event |
| `author_followers` | How many followers the author has (current) |
| `author_following` | How many people the author follows |
| `author_post_count` | Total posts the author has ever made |
| `author_created_at` | When the author created their Bluesky account |
| `author_account_age_days` | Account age in days at the time of posting |

---
## Section 2: Getting to Know the Data

Before doing any causal inference, we need to understand what we're working with. This is like a doctor taking vitals before diagnosing anything.

In [ ]:
# How many posts do we have in each group?
print("Posts by event and period:")
print(df.groupby(['event', 'period']).size().unstack(fill_value=0))
print()
print("Posts by topic group:")
print(df['topic_group'].value_counts())

In [ ]:
# What does engagement look like?
# Social media engagement is famously skewed: most posts get very little,
# a few posts go viral. Let's see if that's true here.

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(['like_count', 'repost_count', 'reply_count']):
    ax = axes[i]
    # Use log scale because of the skew
    data = df[col][df[col] > 0]  # only posts with at least 1
    ax.hist(np.log10(data), bins=40, color='steelblue', edgecolor='white')
    ax.set_xlabel(f'log10({col})')
    ax.set_ylabel('Number of posts')
    ax.set_title(f'{col}\n(median={df[col].median():.0f}, mean={df[col].mean():.1f})')

plt.suptitle("Engagement is heavily skewed: most posts get few likes, a few go viral",
             y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

**Why this matters for analysis:** When data is this skewed, the *mean* is pulled up by viral outliers. A single post with 9,000 likes can shift the average for an entire group. We'll need to think about whether to use means, medians, or log-transformed values in our analysis. (This is a real methodological choice researchers face with social media data.)

In [ ]:
# Who are the authors?
# Let's look at account age: how old were accounts when they posted?

fig, ax = plt.subplots(figsize=(10, 5))
age = df['author_account_age_days'].dropna()
age_valid = age[(age >= 0) & (age < 3000)]  # filter out weird values

ax.hist(age_valid, bins=60, color='coral', edgecolor='white')
ax.axvline(x=0, color='black', linestyle='--', alpha=0.5, label='Brand new account')
ax.set_xlabel('Account age (days) at time of posting')
ax.set_ylabel('Number of posts')
ax.set_title('Many posts come from very new accounts (the migration wave)')
ax.legend()
plt.tight_layout()
plt.show()

# What fraction of authors created their account in the 30 days before their post?
new_accounts = ((age >= 0) & (age <= 30)).sum()
print(f"\n{new_accounts:,} posts ({100*new_accounts/len(age):.1f}%) came from accounts less than 30 days old")

---
## Section 3: Visualizing Engagement

Now let's look at the patterns that motivate our causal questions. The best causal inference starts with a clear picture of what's happening in the raw data.

In [ ]:
# Average likes by topic and period, for the election event
election = df[df['event'] == 'election_2024']

means = election.groupby(['topic_group', 'period'])['like_count'].mean().unstack()
means = means[['before', 'after']]  # order columns

ax = means.plot(kind='bar', figsize=(10, 5), color=['#4878CF', '#E24A33'])
ax.set_ylabel('Average likes per post')
ax.set_xlabel('')
ax.set_title('Average likes before vs. after the 2024 election, by topic')
ax.legend(title='Period')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Question 1:** Look at the bar chart above. Did likes go up after the election for *all* topics, or just political posts? Why might this matter for causal inference?

*Hint: If likes went up for every topic equally, that would suggest something changed about the platform as a whole (e.g., more users = more likes for everyone). If the increase was larger for political posts, that suggests something specific about political content changed.*

**Your answer:**

In [ ]:
# Let's also look at the follower distribution
# This will matter for the RDD section later

fig, ax = plt.subplots(figsize=(10, 5))

followers = df['author_followers']
# Cap at 10,000 for visualization (long tail beyond that)
followers_capped = followers[followers <= 10000]

ax.hist(followers_capped, bins=80, color='mediumseagreen', edgecolor='white')
ax.set_xlabel('Follower count')
ax.set_ylabel('Number of posts')
ax.set_title('Most authors have few followers, but some have thousands')

# Mark some round numbers
for threshold in [100, 500, 1000, 5000]:
    ax.axvline(x=threshold, color='red', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Median followers: {followers.median():.0f}")
print(f"Mean followers: {followers.mean():.0f}")
print(f"Max followers: {followers.max():,.0f}")

---
## Section 4: Difference-in-Differences (DiD)

### The idea

We want to know: **did the 2024 election cause political content on Bluesky to get more engagement?**

We can't just compare political posts before vs. after the election, because *everything* on Bluesky might have changed (the platform was growing rapidly, more users means more likes for everyone). A simple before/after comparison would confound the election effect with the platform growth effect.

**Difference-in-Differences** solves this by using a **control group**. The logic:

1. Political posts are the **treated group** (directly affected by the election)
2. Non-political posts (tech, sports, culture) are the **control group** (affected by platform growth, but not by the election itself)
3. The **first difference**: how much did likes change for political posts? (before vs. after)
4. The **second difference**: how much did likes change for the control group? (before vs. after)
5. The **difference-in-differences**: subtract #4 from #3. This removes the platform-wide trend, isolating the election's specific effect on political content.

### The key assumption

DiD only works if the treated and control groups would have followed **parallel trends** in the absence of treatment. In our case: if there had been no election, political and non-political posts would have experienced the same change in engagement. We can't prove this, but we can check whether it seems plausible.

### Step 1: Build the 2x2 table

The classic DiD setup is a 2x2 table of group means. Let's build it.

In [ ]:
# Focus on the election event
election = df[df['event'] == 'election_2024'].copy()

# Create a binary treatment variable:
# treated = 1 if the post is about politics, 0 otherwise
election['treated'] = (election['topic_group'] == 'politics').astype(int)

# Create a binary post-event variable:
# post = 1 if the post was written after the election, 0 if before
election['post'] = (election['period'] == 'after').astype(int)

# Build the 2x2 table of mean likes
table = election.groupby(['treated', 'post'])['like_count'].mean().unstack()
table.index = ['Control (non-political)', 'Treated (political)']
table.columns = ['Before election', 'After election']

print("Average likes per post:")
print(table.round(2))
print()

# Compute the DiD estimate by hand
diff_treated = table.loc['Treated (political)', 'After election'] - table.loc['Treated (political)', 'Before election']
diff_control = table.loc['Control (non-political)', 'After election'] - table.loc['Control (non-political)', 'Before election']
did_estimate = diff_treated - diff_control

print(f"Change for political posts:     {diff_treated:+.2f} likes")
print(f"Change for non-political posts:  {diff_control:+.2f} likes")
print(f"")
print(f"Difference-in-Differences:       {did_estimate:+.2f} likes")
print(f"")
print("Interpretation: after removing the platform-wide trend,")
print(f"the election caused political posts to get about {did_estimate:.1f} more likes on average.")

### Step 2: The regression version

The 2x2 table gives us the DiD estimate, but it doesn't tell us whether the result is **statistically significant** (could it be due to chance?) or let us **control for other variables**.

The standard DiD regression is:

$$Y_i = \beta_0 + \beta_1 \cdot \text{Treated}_i + \beta_2 \cdot \text{Post}_i + \beta_3 \cdot (\text{Treated}_i \times \text{Post}_i) + \varepsilon_i$$

where:
- $\beta_1$ captures the baseline difference between political and non-political posts
- $\beta_2$ captures the overall time trend (platform growth affecting everyone)
- $\beta_3$ is **the DiD estimate**: the *additional* change for political posts after the election

This $\beta_3$ should match the number we computed by hand above.

In [ ]:
# Run the DiD regression
# The formula "treated * post" automatically includes treated, post,
# AND their interaction (treated:post). The interaction is our DiD estimate.

model = smf.ols('like_count ~ treated * post', data=election).fit()

print(model.summary().tables[1])
print()
print("Read the table above:")
print(f"  - Intercept ({model.params['Intercept']:.2f}): avg likes for control group, before election")
print(f"  - treated ({model.params['treated']:.2f}): political posts start with this many more likes than control")
print(f"  - post ({model.params['post']:.2f}): all posts got this many more likes after the election (platform trend)")
print(f"  - treated:post ({model.params['treated:post']:.2f}): the DiD estimate! The EXTRA change for political posts")
print(f"  - p-value for DiD: {model.pvalues['treated:post']:.4f}")

### Step 3: Adding controls

One concern: maybe the change in political engagement isn't about *politics* specifically, but about the *type of authors* who post about politics. Perhaps political posters just have more followers, and more followers means more likes regardless.

We can add controls for author characteristics to see if the DiD estimate survives.

In [ ]:
# Add log(followers + 1) as a control
# We add 1 to avoid log(0), which is undefined
election['log_followers'] = np.log1p(election['author_followers'])

# DiD with controls
model_controls = smf.ols(
    'like_count ~ treated * post + log_followers + is_reply + has_link + has_image',
    data=election
).fit()

print("DiD with author/post controls:")
print(model_controls.summary().tables[1])
print()
print(f"DiD estimate without controls: {model.params['treated:post']:.2f}")
print(f"DiD estimate with controls:    {model_controls.params['treated:post']:.2f}")
print()
print("Does the estimate change much when we add controls?")
print("If not, that's reassuring: the result isn't driven by differences in who posts about politics.")

**Question 2:** The DiD estimate tells us how much *more* engagement political posts got after the election, compared to what we'd expect from the platform-wide trend alone. In your own words, what is the causal claim we're making? What are two reasons this estimate might be wrong (i.e., two ways the parallel trends assumption could fail)?

**Your answer:**

### Step 4: Visualize the DiD

A good DiD plot shows the before/after change for both groups. If the lines are parallel before the event and diverge after, that supports our causal story.

In [ ]:
# DiD visualization
fig, ax = plt.subplots(figsize=(8, 5))

# Compute means for each group x period
groups = {
    'Political posts': election[election['treated'] == 1],
    'Non-political posts': election[election['treated'] == 0],
}

colors = {'Political posts': '#E24A33', 'Non-political posts': '#4878CF'}
x_positions = [0, 1]  # before=0, after=1

for label, group_df in groups.items():
    before_mean = group_df[group_df['post'] == 0]['like_count'].mean()
    after_mean = group_df[group_df['post'] == 1]['like_count'].mean()
    ax.plot(x_positions, [before_mean, after_mean],
            'o-', color=colors[label], markersize=10, linewidth=2.5, label=label)

# Add a dashed line showing where political posts WOULD have been
# under parallel trends (control trend applied to treated baseline)
ctrl_before = groups['Non-political posts'][groups['Non-political posts']['post'] == 0]['like_count'].mean()
ctrl_after = groups['Non-political posts'][groups['Non-political posts']['post'] == 1]['like_count'].mean()
treat_before = groups['Political posts'][groups['Political posts']['post'] == 0]['like_count'].mean()
counterfactual = treat_before + (ctrl_after - ctrl_before)

ax.plot([0, 1], [treat_before, counterfactual],
        '--', color='#E24A33', alpha=0.4, linewidth=2,
        label='Political (counterfactual)')

# Annotate the DiD
ax.annotate('', xy=(1.05, groups['Political posts'][groups['Political posts']['post'] == 1]['like_count'].mean()),
            xytext=(1.05, counterfactual),
            arrowprops=dict(arrowstyle='<->', color='black', lw=1.5))
ax.text(1.12, (counterfactual + groups['Political posts'][groups['Political posts']['post'] == 1]['like_count'].mean()) / 2,
        f'DiD = {did_estimate:.1f}', fontsize=11, va='center')

ax.set_xticks(x_positions)
ax.set_xticklabels(['Before election', 'After election'])
ax.set_ylabel('Average likes per post')
ax.set_title('Difference-in-Differences: Election Effect on Political Engagement')
ax.legend(loc='upper left')
ax.axvline(x=0.5, color='gray', linestyle=':', alpha=0.3)
plt.tight_layout()
plt.show()

### Step 5: Robustness check with log-transformed outcome

Remember that engagement is heavily skewed. A few viral posts could be driving our result. Let's re-run the DiD using log(likes + 1) as the outcome. If the result holds, it's not just about outliers.

In [ ]:
# Log-transform the outcome
election['log_likes'] = np.log1p(election['like_count'])

model_log = smf.ols('log_likes ~ treated * post', data=election).fit()

print("DiD with log(likes + 1) as outcome:")
print(model_log.summary().tables[1])
print()
print(f"DiD estimate (log scale): {model_log.params['treated:post']:.4f}")
print(f"p-value: {model_log.pvalues['treated:post']:.4f}")
print()
print("On the log scale, the coefficient is approximately the percentage change.")
print(f"So political posts got roughly {100*model_log.params['treated:post']:.1f}% more likes")
print("relative to what we'd expect from the platform-wide trend.")

**Question 3:** Compare the results from the levels regression (Step 2) and the log regression (Step 5). Do they tell the same story? Why might researchers prefer the log specification for social media data?

**Your answer:**

---
## Section 5: Regression Discontinuity Design (RDD)

### The idea

DiD compares groups over time. RDD takes a completely different approach: it looks for **jumps** at a threshold.

On social media, a natural question is: **do accounts that cross a follower milestone (like 1,000 followers) suddenly get more engagement?**

There are reasons this could happen:
- Platforms might algorithmically boost accounts that hit certain thresholds
- Users might perceive accounts with round follower numbers as more credible
- Authors who hit milestones might change their behavior (post more confidently, etc.)

The RDD logic: compare posts from authors with *just below* 1,000 followers to posts from authors with *just above* 1,000 followers. These authors are nearly identical in every way, except one group crossed the threshold. Any jump in engagement right at the cutoff is evidence of a causal effect.

### Important caveat

For RDD to work, people can't precisely *manipulate* their position relative to the cutoff. If authors deliberately buy followers to cross 1,000, the groups on either side won't be comparable. We'll check for this.

### Step 1: Choose a cutoff and look at the raw data

Let's start by looking at whether there's a visible jump in engagement around 1,000 followers. A **bin scatter** averages the outcome within small bins of the running variable, making it easy to see patterns.

In [ ]:
# RDD around 1,000 followers
CUTOFF = 1000
BANDWIDTH = 500  # look at authors with 500-1500 followers

# Filter to a window around the cutoff
rdd_df = df[(df['author_followers'] >= CUTOFF - BANDWIDTH) &
            (df['author_followers'] <= CUTOFF + BANDWIDTH)].copy()

print(f"Posts in the window ({CUTOFF - BANDWIDTH} to {CUTOFF + BANDWIDTH} followers): {len(rdd_df):,}")
print(f"  Below cutoff: {(rdd_df['author_followers'] < CUTOFF).sum():,}")
print(f"  Above cutoff: {(rdd_df['author_followers'] >= CUTOFF).sum():,}")

# Create the running variable (distance from cutoff)
rdd_df['running'] = rdd_df['author_followers'] - CUTOFF
rdd_df['above'] = (rdd_df['running'] >= 0).astype(int)

# Bin scatter
n_bins = 30
rdd_df['bin'] = pd.cut(rdd_df['running'], bins=n_bins)
bin_means = rdd_df.groupby('bin').agg(
    mean_likes=('like_count', 'mean'),
    mean_followers=('running', 'mean'),
    count=('like_count', 'size')
).dropna()

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#4878CF' if x < 0 else '#E24A33' for x in bin_means['mean_followers']]
ax.scatter(bin_means['mean_followers'], bin_means['mean_likes'],
           c=colors, s=bin_means['count']*2, alpha=0.7, edgecolors='white')
ax.axvline(x=0, color='black', linestyle='--', alpha=0.5, label=f'Cutoff ({CUTOFF} followers)')
ax.set_xlabel(f'Followers relative to {CUTOFF}')
ax.set_ylabel('Average likes per post')
ax.set_title(f'Bin scatter: engagement around {CUTOFF:,} followers')
ax.legend()
plt.tight_layout()
plt.show()

### Step 2: Estimate the discontinuity

The simplest RDD estimate fits separate linear regressions on each side of the cutoff. The jump at the cutoff is our estimate of the causal effect.

In [ ]:
# RDD regression: separate slopes on each side of the cutoff
# Y = b0 + b1*running + b2*above + b3*running*above
# b2 is our RDD estimate (the jump at the cutoff)

rdd_model = smf.ols('like_count ~ running * above', data=rdd_df).fit()

print("RDD regression results:")
print(rdd_model.summary().tables[1])
print()
print(f"Estimated jump at {CUTOFF:,} followers: {rdd_model.params['above']:.2f} likes")
print(f"p-value: {rdd_model.pvalues['above']:.4f}")
print()
if rdd_model.pvalues['above'] < 0.05:
    print(f"Statistically significant! Crossing {CUTOFF:,} followers is associated with")
    print(f"about {rdd_model.params['above']:.1f} more likes per post.")
else:
    print(f"Not statistically significant at the 5% level.")
    print(f"We can't conclude that crossing {CUTOFF:,} followers causes a jump in engagement.")

### Step 3: Validity check (McCrary density test)

A key RDD assumption is that people can't precisely manipulate their position relative to the cutoff. If authors strategically inflate their follower count to get past 1,000, we'd see a **bunching** of authors just above the cutoff and a gap just below.

Let's check the density of authors around the cutoff.

In [ ]:
# McCrary-style density check: is there bunching at the cutoff?
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(rdd_df['running'], bins=50, color='steelblue', edgecolor='white')
ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label=f'Cutoff ({CUTOFF} followers)')
ax.set_xlabel(f'Followers relative to {CUTOFF}')
ax.set_ylabel('Number of posts')
ax.set_title('Density of posts around the cutoff (checking for manipulation)')
ax.legend()
plt.tight_layout()
plt.show()

print("If the histogram looks smooth through the cutoff (no big jump or gap),")
print("that's evidence against manipulation. If there's a suspicious spike")
print("just above the cutoff, that's a red flag.")

### Step 4: Bandwidth sensitivity

A good RDD result should be robust to the choice of bandwidth. If the result only appears with one specific bandwidth, that's suspicious. Let's check.

In [ ]:
# Try different bandwidths and see if the estimate is stable
bandwidths = [200, 300, 400, 500, 600, 800, 1000]
results = []

for bw in bandwidths:
    subset = df[(df['author_followers'] >= CUTOFF - bw) &
                (df['author_followers'] <= CUTOFF + bw)].copy()
    subset['running'] = subset['author_followers'] - CUTOFF
    subset['above'] = (subset['running'] >= 0).astype(int)

    try:
        m = smf.ols('like_count ~ running * above', data=subset).fit()
        results.append({
            'bandwidth': bw,
            'estimate': m.params['above'],
            'se': m.bse['above'],
            'pvalue': m.pvalues['above'],
            'n': len(subset)
        })
    except Exception:
        pass

results_df = pd.DataFrame(results)
print("RDD estimates across bandwidths:")
print(results_df.to_string(index=False, float_format='%.2f'))

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(results_df['bandwidth'], results_df['estimate'],
            yerr=1.96*results_df['se'], fmt='o-', capsize=5, color='steelblue')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Bandwidth (followers on each side of cutoff)')
ax.set_ylabel('RDD estimate (jump in likes)')
ax.set_title('Bandwidth sensitivity: is the RDD estimate stable?')
plt.tight_layout()
plt.show()

**Question 4:** Based on the bin scatter, the regression estimate, the density check, and the bandwidth sensitivity plot: is there evidence that crossing 1,000 followers causes a jump in engagement? What are the strengths and weaknesses of this RDD design?

*Hint: Think about what "1,000 followers" actually means on Bluesky. Is it a threshold the platform uses for anything? Is it just a round number? Does that matter for the causal interpretation?*

**Your answer:**

---
## Section 6: Your Turn

Choose **one** of the following exercises. Write code, produce at least one figure, and explain your findings in 3-5 sentences.

### Option A: Placebo test for DiD

We ran DiD on the **election event**. Now run the same analysis on the **public launch event** (Feb 6, 2024). Bluesky opening to the public shouldn't have specifically boosted political content the way an election would. If we find a large DiD estimate for the public launch too, that's a red flag for our election analysis (it suggests our "control group" isn't really a good control).

*Starter code:*

In [ ]:
# Option A: Placebo test
# Filter to the public launch event and run the same DiD as Section 4

launch = df[df['event'] == 'public_launch'].copy()
launch['treated'] = (launch['topic_group'] == 'politics').astype(int)
launch['post'] = (launch['period'] == 'after').astype(int)

# YOUR CODE HERE: build the 2x2 table, run the regression, make the plot
# Compare the DiD estimate to what we found for the election


### Option B: DiD by topic

Instead of grouping all non-political posts together as "control," run separate DiD analyses using each control group individually (tech, sports, culture). Do you get different estimates? Which control group is most convincing and why?

*Starter code:*

In [ ]:
# Option B: DiD with different control groups
election = df[df['event'] == 'election_2024'].copy()

for control_topic in ['tech', 'sports', 'culture']:
    subset = election[election['topic_group'].isin(['politics', control_topic])].copy()
    subset['treated'] = (subset['topic_group'] == 'politics').astype(int)
    subset['post'] = (subset['period'] == 'after').astype(int)

    # YOUR CODE HERE: run the DiD regression for each control group
    # Print the DiD estimate and p-value
    # Which control group gives the largest/smallest estimate?


### Option C: Try a different RDD cutoff

We used 1,000 followers, but that was arbitrary. Try other cutoffs (100, 500, 5000) and see if any of them show a more convincing discontinuity. You can also try different outcome variables (reposts, replies instead of likes).

*Starter code:*

In [ ]:
# Option C: Try different RDD cutoffs
# Modify the CUTOFF and BANDWIDTH values from Section 5
# and re-run the analysis

for cutoff in [100, 500, 5000]:
    bw = cutoff // 2  # bandwidth = half the cutoff
    subset = df[(df['author_followers'] >= cutoff - bw) &
                (df['author_followers'] <= cutoff + bw)].copy()
    subset['running'] = subset['author_followers'] - cutoff
    subset['above'] = (subset['running'] >= 0).astype(int)

    # YOUR CODE HERE: run the RDD regression, make bin scatter plots
    # Which cutoff shows the most convincing discontinuity?
    print(f"Cutoff = {cutoff:,}, n = {len(subset):,}")


### Option D: The migration effect

The election didn't just change what people talked about; it changed *who was on the platform*. Using `author_account_age_days`, separate posts into those by "veterans" (accounts older than 180 days) and "newcomers" (accounts younger than 30 days). Do newcomers get different engagement than veterans? Does this differ by topic?

*Starter code:*

In [ ]:
# Option D: Veterans vs. newcomers
election = df[df['event'] == 'election_2024'].copy()

election['user_type'] = 'middle'
election.loc[election['author_account_age_days'] > 180, 'user_type'] = 'veteran'
election.loc[election['author_account_age_days'] <= 30, 'user_type'] = 'newcomer'

# YOUR CODE HERE:
# 1. Compare average engagement (likes) for veterans vs. newcomers
# 2. Break this down by topic_group
# 3. Make a grouped bar chart
# 4. What does this tell us about the migration wave?


---

## About this dataset

This dataset was collected from the Bluesky public API in March 2026 for educational use.  Posts were found by searching for topic-specific keywords within date windows around each event. Author profiles were enriched using public profile metadata.

All data was publicly posted on Bluesky. No private messages, follower lists, or non-public information was accessed.

**Columns collected:** post URI, author DID and handle, post text (truncated to 280 chars), timestamp, like/repost/reply counts, post features (links, images, replies, quotes), topic labels, event and period labels, and author metadata (follower/following counts, account creation date, account age at time of posting).